In [1]:
import subprocess, sys, torch

print(sys.version.split()[0])
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout[:600])

3.12.13
torch: 2.10.0+cu128
cuda available: True
gpu: Tesla T4
vram: 15.6
Sun Aug  9 05:10:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|===============


In [2]:
import pandas as pd
import os

EVAL_BASE = "/kaggle/input/datasets/mhashamhussain/vmc2026-track2-eval/vmc2026_track2_eval_phase_distro"

meta = pd.read_csv(f"{EVAL_BASE}/metadata.csv", sep="|", header=None,
                    names=["filename", "target_emotion", "transcript"])
disk = set(os.listdir(f"{EVAL_BASE}/wav"))

avail = meta[meta.filename.isin(disk)].reset_index(drop=True)

print("metadata rows:", len(meta))
print("wavs on disk:", len(disk))
print("usable rows:", len(avail))
assert len(avail) == len(disk), "mismatch between disk files and matched metadata rows"
print(avail.head(3))

metadata rows: 2731
wavs on disk: 2574
usable rows: 2574
                                  filename target_emotion  \
0  vmc2026-track2-sys007-spk017-utt127.wav            Sad   
1  vmc2026-track2-sys013-spk011-utt151.wav       Surprise   
2  vmc2026-track2-sys010-spk023-utt044.wav          Angry   

                                     transcript  
0  Rabbit gave dog a hurrying up sort of nudge.  
1              Take courage all isn't lost yet.  
2                      Monster made a deep bow.  


In [3]:
!pip install git+https://github.com/sarulab-speech/UTMOSv2.git --quiet

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
import utmosv2
print(utmosv2.__file__)
import torch
print("torch after install:", torch.__version__)
print("cuda still available:", torch.cuda.is_available())

/usr/local/lib/python3.12/dist-packages/utmosv2/__init__.py
torch after install: 2.10.0+cu128
cuda still available: True


In [5]:
import time

t0 = time.time()
model = utmosv2.create_model(pretrained=True)
print("load time (s):", round(time.time() - t0, 1))
print(type(model))
print(next(model.parameters()).device)

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/86.5M [00:00<?, ?B/s]

--2026-08-09 05:10:50--  https://huggingface.co/sarulab-speech/UTMOSv2/resolve/main/fold0_s42_best_model.pth
Resolving huggingface.co (huggingface.co)... 13.226.251.81, 13.226.251.112, 13.226.251.66, ...
Connecting to huggingface.co (huggingface.co)|13.226.251.81|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/67bf20260fd2e346e4c81864/149175e37e848270ca81a6f11f621dfd71388e2e3f3cfd369634c181adfca88a?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27fold0_s42_best_model.pth%3B+filename%3D%22fold0_s42_best_model.pth%22%3B&user_id=public&X-Xet-Cas-Uid=public&Expires=1786255850&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjdiZjIwMjYwZmQyZTM0NmU0YzgxODY0LzE0OTE3NWUzN2U4NDgyNzBjYTgxYTZmMTFmNjIxZGZkNzEzODhlMmUzZjNjZmQzNjk2MzRjMTgxYWRmY2E4OGFcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomdXNlcl9pZD1wdWJsaWMmWC1YZXQtQ2FzLVVpZD1wdWJsaWMiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3N

Done.
Loaded checkpoint from /root/.cache/utmosv2/models/fusion_stage3/fold0_s42_best_model.pth
load time (s): 24.4
<class 'utmosv2._core.model._models.UTMOSv2Model'>
cpu


In [6]:
model = model.to("cuda")
print(next(model.parameters()).device)

cuda:0


In [7]:
import librosa

sample_files = avail.filename.head(3).tolist()

for fn in sample_files:
    path = f"{EVAL_BASE}/wav/{fn}"
    y, sr = librosa.load(path, sr=16000, mono=True)
    pred = model.predict(data=y, sr=16000)
    print(fn, "->", pred, type(pred))

Predicting:   0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/utmosv2/_core/model/_common.py:290: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Predicting: 100%|██████████| 1/1 [00:06<00:00,  6.18s/it]


vmc2026-track2-sys007-spk017-utt127.wav -> [3.598] <class 'numpy.ndarray'>


Predicting: 100%|██████████| 1/1 [00:01<00:00,  1.26s/it]


vmc2026-track2-sys013-spk011-utt151.wav -> [2.816] <class 'numpy.ndarray'>


Predicting: 100%|██████████| 1/1 [00:01<00:00,  1.46s/it]

vmc2026-track2-sys010-spk023-utt044.wav -> [3.02] <class 'numpy.ndarray'>


In [ ]:
import json
from tqdm.auto import tqdm

CKPT_PATH = "/kaggle/working/qmos_checkpoint.json"

preds = {}
failed = []

for fn in tqdm(avail.filename, total=len(avail)):
    try:
        path = f"{EVAL_BASE}/wav/{fn}"
        y, sr = librosa.load(path, sr=16000, mono=True)
        pred = model.predict(data=y, sr=16000)
        preds[fn] = float(pred[0])
    except Exception as e:
        failed.append((fn, repr(e)))

    if len(preds) % 250 == 0 and len(preds) > 0:
        with open(CKPT_PATH, "w") as f:
            json.dump(preds, f)

with open(CKPT_PATH, "w") as f:
    json.dump(preds, f)

print("succeeded:", len(preds))
print("failed:", len(failed))
print(failed[:5])

  0%|          | 0/2574 [00:00<?, ?it/s]


Predicting: 100%|██████████| 1/1 [00:01<00:00,  1.25s/it]

Predicting: 100%|██████████| 1/1 [00:01<00:00,  1.38s/it]

Predicting: 100%|██████████| 1/1 [00:01<00:00,  1.27s/it]

Predicting: 100%|██████████| 1/1 [00:01<00:00,  1.16s/it]

Predicting: 100%|██████████| 1/1 [00:01<00:00,  1.38s/it]

Predicting: 100%|██████████| 1/1 [00:01<00:00,  1.20s/it]

Predicting: 100%|██████████| 1/1 [00:01<00:00,  1.33s/it]

Predicting: 100%|██████████| 1/1 [00:01<00:00,  1.17s/it]

Predicting: 100%|██████████| 1/1 [00:01<00:00,  1.14s/it]

Predicting: 100%|██████████| 1/1 [00:01<00:00,  1.13s/it]

Predicting: 100%|██████████| 1/1 [00:01<00:00,  1.18s/it]

Predicting: 100%|██████████| 1/1 [00:01<00:00,  1.33s/it]

Predicting: 100%|██████████| 1/1 [00:01<00:00,  1.30s/it]

Predicting: 100%|██████████| 1/1 [00:01<00:00,  1.17s/it]

Predicting: 100%|██████████| 1/1 [00:01<00:00,  1.15s/it]

Predicting: 100%|██████████| 1/1 [00:01<00:00,  1.16s/it]

Predicting: 100%|██████████| 1/1 [00:01<00:00,  1.34s/i

In [10]:
import os
print(os.listdir("/kaggle/working"))

['.virtual_documents', 'qmos_checkpoint.json']


In [11]:
print("succeeded:", len(preds))
print("failed:", len(failed))

succeeded: 2574
failed: 0


# Training Data (Future Work)

In [ ]:
import os
import pandas as pd

TRAIN_BASE = "/kaggle/input/datasets/mhashamhussain/vmc2026-track2-train/vmc2026-track2"

train = pd.read_csv(f"{TRAIN_BASE}/sets/train.csv", sep="|")
dev   = pd.read_csv(f"{TRAIN_BASE}/sets/dev.scp", header=None, names=["wavID"])
disk_train = set(os.listdir(f"{TRAIN_BASE}/wav"))

print("rating rows:", len(train))
print("unique wavs rated:", train.wavID.nunique())
print("unique listeners:", train.lisID.nunique())
print("dev.scp entries:", len(dev))
print("wavs on disk:", len(disk_train))
print("expected total:", train.wavID.nunique() + len(dev))
print("ratings per clip:", round(len(train) / train.wavID.nunique(), 1))
print(train.dtypes)